# News Forecast Pipeline — 3 крупнейших СМИ РФ

**Цель:** Сбор → ETL → Анализ → Бэктест → Прогноз на **02.04.2026**

**СМИ:** Коммерсантъ, Лента.ру, Интерфакс

| Шаг | Ячейка | Описание |
|-----|--------|----------|
| 0 | Setup | Настройка окружения и импорты |
| 1 | Scrape | Сбор данных за 90 дней (RSS + архив) |
| 2 | ETL | Очистка, дедупликация, фильтрация |
| 3 | Analyze | Топики, частоты, NER, noise check |
| 4 | Backtest | Holdout-проверка на последних 7 днях |
| 5 | Metrics | Оценка качества прогноза |
| 6 | Forecast | Прогноз на целевую дату |
| 7 | Results | Просмотр итогового JSON |

---
## Ячейка 0 — Setup

In [1]:
import sys, os, datetime
from dotenv import load_dotenv

# 1. Добавляем текущую директорию в путь импорта
sys.path.insert(0, os.getcwd())

# 2. Загружаем переменные окружения (.env)
load_dotenv()

import config, scraper, etl, analyzer, forecaster, backtester, metrics

print(f"Python: {sys.version.split()[0]}")
print(f"Рабочая директория: {os.getcwd()}")
print(f"Целевая дата прогноза: {config.TARGET_DATE}")

# Проверка наличия API ключа
if os.getenv('OPENROUTER_API_KEY'):
    print("✅ OpenRouter API Key: найден")
else:
    print("⚠️  OpenRouter API Key: НЕ НАЙДЕН (проверьте .env)")

Python: 3.11.13
Рабочая директория: /Users/justcomex/Documents/nstu/pet/titles_forecating
Целевая дата прогноза: 2026-04-02
✅ OpenRouter API Key: найден


---
## Ячейка 1 — Scrape: сбор данных

> ⏱️ **~5-20 минут** в зависимости от скорости соединения и количества доступных архивных страниц.
> 
> Можно ограничить список СМИ через `OUTLETS_TO_SCRAPE` или уменьшить `START_DATE`.
> 
> Добавьте `enrich_leads=True` для полного скрапинга лидов (медленнее, по 1-2 сек/статья).

In [2]:
import config, os, json
from scraper import scrape_all

# ── Настройки скрапинга ──────────────────────────────────────────
OUTLETS_TO_SCRAPE = config.OUTLET_SLUGS      # все 3, или например ["kommersant", "lenta"]
SCRAPE_FROM       = config.HISTORY_FROM       # дата начала (по умолчанию: TODAY − 90 дней)
SCRAPE_TO         = config.TODAY
ENRICH_LEADS      = False                    # True = доп. запросы за каждым лидом (медленно)
LOAD_EXISTING     = True                     # True = загрузить из JSON, если он есть
# ────────────────────────────────────────────────────────────────

if LOAD_EXISTING and os.path.exists(config.INTERMEDIATE_JSON):
    print(f"[pipeline] Загружаем существующий скрап из: {config.INTERMEDIATE_JSON}")
    with open(config.INTERMEDIATE_JSON, "r", encoding="utf-8") as f:
        scrape_results = json.load(f)
else:
    print("[pipeline] Файл не найден или LOAD_EXISTING=False. Запускаем скрапинг...")
    scrape_results = scrape_all(
        slugs=OUTLETS_TO_SCRAPE,
        start_date=SCRAPE_FROM,
        end_date=SCRAPE_TO,
        enrich_leads=ENRICH_LEADS,
    )

print("── Итог скрапинга ──")
for slug, recs in scrape_results.items():
    print(f"  {slug:<12} {len(recs):>5} записей")

[pipeline] Загружаем существующий скрап из: /Users/justcomex/Documents/nstu/pet/titles_forecating/data/raw/intermediate_scrape.json
── Итог скрапинга ──
  kommersant     350 записей
  lenta         3699 записей
  interfax      5980 записей


---
## Ячейка 2 — ETL: очистка и дедупликация

In [3]:
import importlib, etl as _etl_mod
importlib.reload(_etl_mod)
from etl import run_etl, load_clean
import pandas as pd

run_etl(slugs=config.OUTLET_SLUGS)

# Сводка по чистым данным
print("\n── Итог ETL ──")
summary_rows = []
for slug in config.OUTLET_SLUGS:
    df = load_clean(slug)
    if df.empty:
        summary_rows.append({"outlet": slug, "records": 0, "with_lead": 0,
                              "date_min": None, "date_max": None})
        continue
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    summary_rows.append({
        "outlet":   slug,
        "records":  len(df),
        "with_lead": df["lead"].notna().sum(),
        "date_min": df["published_at"].min().date() if not df.empty else None,
        "date_max": df["published_at"].max().date() if not df.empty else None,
    })

pd.DataFrame(summary_rows)


  [etl] kommersant: 1397 raw records
  [etl] exact URL dedup: 1397 -> 350
  [etl] near-dedup removed 4 rows (threshold=0.85)
  [etl] kommersant: 346 clean records
  [etl] saved /Users/justcomex/Documents/nstu/pet/titles_forecating/data/clean/kommersant_clean.csv
  [etl] lenta: 7394 raw records
  [etl] exact URL dedup: 7394 -> 3700
  [etl] opinion filter removed 1 rows
  [etl] near-dedup removed 812 rows (threshold=0.85)
  [etl] lenta: 2887 clean records
  [etl] saved /Users/justcomex/Documents/nstu/pet/titles_forecating/data/clean/lenta_clean.csv
  [etl] interfax: 5980 raw records
  [etl] exact URL dedup: 5980 -> 5980
  [etl] opinion filter removed 1 rows
  [etl] near-dedup removed 608 rows (threshold=0.85)
  [etl] interfax: 5371 clean records
  [etl] saved /Users/justcomex/Documents/nstu/pet/titles_forecating/data/clean/interfax_clean.csv

── Итог ETL ──


,outlet,records,with_lead,date_min,date_max
0,kommersant,346,346,2026-03-26,2026-03-29
1,lenta,2887,2887,2025-12-29,2026-03-29
2,interfax,5371,5371,2025-12-29,2026-03-29


In [4]:
# Просмотр нескольких строк для одного СМИ
PREVIEW_OUTLET = "kommersant"   # поменяйте на любой slug

df_preview = load_clean(PREVIEW_OUTLET)
df_preview["published_at"] = pd.to_datetime(df_preview["published_at"], errors="coerce")
df_preview.sort_values("published_at", ascending=False)[["published_at", "rubric", "title", "lead"]].head(10)

,published_at,rubric,title,lead
345,2026-03-29 17:05:00,Мир,Нетаньяху распорядился расширить буферную зону...,Премьер-министр Израиля Биньямин Нетаньяху отд...
344,2026-03-29 16:41:17,Происшествия,Жилой дом в Ленинградской области поврежден пр...,Обломки БПЛА повредили жилой дом в деревне Сис...
343,2026-03-29 16:23:25,Общество,В Махачкале восстановили работу двух из трех п...,Две из трех подтопленных подстанций в Махачкал...
342,2026-03-29 16:01:55,Происшествия,В Энергодаре произошли перебои с электроэнерги...,Энергодар в Запорожской области был обесточен ...
341,2026-03-29 16:00:21,Новости,Главные новости за выходные 28–29 марта,В Чечне ввели режим повышенной готовности из-з...
340,2026-03-29 15:49:03,Мир,Иран заявил об ударе по химзаводу в Израиле,Иран нанес удар по промышленному объекту в рай...
339,2026-03-29 15:46:20,Мир,В офисе Нетаньяху объяснили недопуск патриарха...,Канцелярия премьер-министра Израиля Биньямина ...
338,2026-03-29 15:45:59,Мир,В 16 странах прошли протесты против Трампа и в...,28 марта по всему миру прошла волна массовых п...
337,2026-03-29 15:41:03,Мир,МИД Италии вызвал посла Израиля из-за запрета ...,Полиция Израиля не пустила латинского патриарх...
336,2026-03-29 15:40:51,Происшествия,В Жуковском ликвидировали открытое горение при...,В подмосковном городе Жуковском произошел круп...


---
## Ячейка 3 — Analyze: темы, частоты, NER, noise check

In [5]:
import config
from analyzer import analyze_all

analysis = analyze_all(slugs=config.OUTLET_SLUGS)



[analyze] kommersant: 346 records
  [noise] kommersant: daily_mean=86.5, cv=0.878, stable=False
  [topics] top-5:
    города / дмитрий / песков / сочи / пресс секретарь: 25 (14.7%)
    сша / ирана / трамп / президент / саудовской: 24 (14.1%)
    области / губернатор / региона / бпла / ударе: 16 (9.4%)
    россии / глава / тасс / москве / всу: 15 (8.8%)
    служба / пресс служба / пресс / россии / правительства: 10 (5.9%)
  [entities] top persons: {'Дональд Трамп': 5, 'Саудовской Аравии': 3, 'Порт Салала': 3, 'Дональда Трампа': 2, 'Марко Рубио': 2}
  [style]  avg_title=9.5 words, avg_lead=28.1 words

[analyze] lenta: 2887 records
  [noise] lenta: daily_mean=31.7, cv=0.508, stable=False
  [topics] top-5:
    украине / трампа / трамп / техника / сша: 264 (45.1%)
    сша / ссср / украине / трампа / трамп: 64 (10.9%)
    россии / сша / украине / трампа / трамп: 34 (5.8%)
    жизни / россии / сша / украине / трампа: 28 (4.8%)
    всу / ссср / россии / техника / сша: 28 (4.8%)
  [entities] t

In [6]:
# ── Топ-10 тем по каждому СМИ ────────────────────────────────────
import pandas as pd

TOPIC_OUTLET = "interfax"   # поменяйте на нужный slug

if TOPIC_OUTLET in analysis and analysis[TOPIC_OUTLET]:
    freq = analysis[TOPIC_OUTLET]["topic_freq"]
    print(f"\nТоп-10 тем ({config.OUTLETS[TOPIC_OUTLET]['name']}, последние {config.TOPIC_WINDOW} дней):\n")
    display(freq.head(10)[["cluster_name", "count", "pct"]])
else:
    print(f"Нет данных для {TOPIC_OUTLET} — сначала запустите Scrape+ETL")


Топ-10 тем (Интерфакс, последние 14 дней):



,cluster_name,count,pct
1,трамп / сша / россии / путин / против,468,57.9
3,сша / трамп / россии / путин / против,69,8.5
4,трамп / сша / ирана / россии / путин,29,3.6
13,заявили / сша / ирана / россии / мид,23,2.8
2,области / сша / трамп / россии / путин,21,2.6
12,глава / мид / сша / ирана / против,21,2.6
0,против / ирана / сша / трамп / глава,19,2.4
10,млрд / россии / трамп / сша / млн,18,2.2
9,мид / россии / сша / ирана / против,17,2.1
6,россии / против / трамп / бпла / сша,17,2.1


In [7]:
# ── Noise check: стабильность объёма по всем СМИ ─────────────────
noise_rows = []
for slug, res in analysis.items():
    if not res:
        continue
    n = res["noise"]
    noise_rows.append({
        "outlet":       slug,
        "daily_mean":   n.get("daily_mean"),
        "cv":           n.get("cv"),
        "stable":       "✅" if n.get("stable") else "⚠️",
        "top_rubric":   n.get("top_rubric"),
        "top_share_%":  round(n.get("top_rubric_share", 0) * 100, 1),
        "rubric_noisy": "⚠️" if n.get("rubric_noisy") else "✅",
    })

pd.DataFrame(noise_rows)

,outlet,daily_mean,cv,stable,top_rubric,top_share_%,rubric_noisy
0,kommersant,86.5,0.878,⚠️,Мир,37.0,✅
1,lenta,31.7,0.508,⚠️,,93.0,⚠️
2,interfax,59.0,0.087,✅,,99.5,⚠️


In [8]:
# ── Топ сущностей (персоны, организации) ─────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ENT_OUTLET = "kommersant"   # поменяйте
ENT_TYPE   = "persons"  # persons | orgs | locations

if ENT_OUTLET in analysis and analysis[ENT_OUTLET]:
    ents = analysis[ENT_OUTLET]["entities"].get(ENT_TYPE, pd.Series())
    if not ents.empty:
        fig = go.Figure(go.Bar(
            x=ents.values[:20][::-1],
            y=ents.index[:20][::-1],
            orientation="h"
        ))
        fig.update_layout(
            title=f"Топ упоминаний: {ENT_TYPE} — {config.OUTLETS[ENT_OUTLET]['name']}",
            height=500, margin=dict(l=200)
        )
        fig.show()
    else:
        print(f"Нет сущностей типа '{ENT_TYPE}' для {ENT_OUTLET}")

In [9]:
# ── Динамика публикаций по дням (все СМИ) ────────────────────────
import plotly.express as px
from etl import load_all_clean

df_all = load_all_clean()
if not df_all.empty:
    daily = (
        df_all.groupby([df_all["published_at"].dt.date, "outlet"])
              .size()
              .reset_index(name="count")
    )
    daily.columns = ["date", "outlet", "count"]
    fig = px.line(daily, x="date", y="count", color="outlet",
                  title="Количество публикаций по дням",
                  labels={"count": "Публикаций", "date": "Дата"})
    fig.show()
else:
    print("Нет данных — запустите Scrape + ETL")

---
## Ячейка 4 — Backtest: holdout-проверка

> Обучение на `[all data − 7 days]`, тест на последних 7 днях.
> Методы: **inertia**, **frequency**, **calendar**.

In [10]:
import config
from backtester import backtest_all

# ── Настройки ────────────────────────────────────────────────────
BT_OUTLETS     = config.OUTLET_SLUGS
BT_HOLDOUT     = config.BACKTEST_DAYS      # 7 дней
BT_METHODS     = ["inertia", "frequency", "calendar", "llm"]
# ────────────────────────────────────────────────────────────────

bt_results = backtest_all(
    slugs=BT_OUTLETS,
    holdout_days=BT_HOLDOUT,
    methods=BT_METHODS,
)

print("\n── Бэктест завершён ──")
for slug, days in bt_results.items():
    print(f"  {slug:<12} {len(days)} дней")



[backtest] kommersant: holdout 2026-03-23 → 2026-03-29
  [llm] kommersant | topic: россии / сша / сергей лавров / сергей / путин...
  [llm] kommersant | topic: лавров / сергей лавров / сергей / путин / сша...
  2026-03-27: 190 actual articles
  [llm] kommersant | topic: иране / компаний / прокуратуры / александр / своем...
  [llm] kommersant | topic: вице / сочи / премьер / правительства / россии...
  2026-03-28: 81 actual articles
  [llm] kommersant | topic: россии / сша / тыс / служба / власти...
  [llm] kommersant | topic: области / губернатор / бпла / региона / ударе...
  2026-03-29: 67 actual articles
  [backtest] saved → /Users/justcomex/Documents/nstu/pet/titles_forecating/data/forecasts/backtest_kommersant_20260329.json

[backtest] lenta: holdout 2026-03-23 → 2026-03-29
  [llm] lenta | topic: украине / трампа / трамп / техника / сша...
  [llm] lenta | topic: сша / ссср / украине / трампа / трамп...
  2026-03-23: 30 actual articles
  [llm] lenta | topic: украине / трампа / трам

In [11]:
# Просмотр одного дня бэктеста
import json

BT_OUTLET = "kommersant"
BT_DAY_IDX = 0    # индекс дня в holdout (0 = самый ранний)

if BT_OUTLET in bt_results and bt_results[BT_OUTLET]:
    day = bt_results[BT_OUTLET][BT_DAY_IDX]
    print(f"Дата: {day['date']}")
    print(f"\nФактические заголовки ({len(day['actual']['titles'])})")
    for t in day["actual"]["titles"][:10]:
        print(f"  ► {t}")
    print(f"\nПрогноз 'frequency' (топ-5 тем)")
    for p in day["predictions"].get("frequency", [])[:5]:
        print(f"  • {p.get('topic_label', '')[:80]}")
else:
    print(f"Нет данных бэктеста для {BT_OUTLET} — запустите ячейку выше")

Дата: 2026-03-27

Фактические заголовки (190)
  ► Трамп дал еще одну отсрочку Ирану
  ► Подпись Трампа появится на долларах США в честь 250-летия страны
  ► Суд встал на сторону Anthropic в споре с Пентагоном
  ► WSJ: Пентагон обсуждает переброску еще 10 тыс. военнослужащих на Ближний Восток
  ► Полиция Словакии начала расследование против Фицо по подозрению в госизмене
  ► Россия запросила консультации СБ ООН из-за ударов по гражданским объектам Ирана
  ► Москва и Кабул начали обсуждать привлечение афганцев для работы в России
  ► В Карибском море пропали два мексиканских судна с гуманитарной помощью для Кубы
  ► Дроны атаковали промзону в Череповце
  ► Сенаторы США готовят санкции против Венгрии из-за блокировки помощи Украине

Прогноз 'frequency' (топ-5 тем)
  • россии / сша / сергей лавров / сергей / путин
  • лавров / сергей лавров / сергей / путин / сша
  • сша / сергей лавров / сергей / россии / путин
  • сша / сергей лавров / сергей / россии / путин
  • путин / сша / сергей лав

---
## Ячейка 5 — Metrics: оценка качества

| Метрика | Целевое значение | Описание |
|---------|-----------------|----------|
| topic_hit_rate | ≥ 0.30 | Совпадение тематики (Jaccard) |
| entity_match_f1 | ≥ 0.20 | Совпадение персон и орг (F1) |
| semantic_similarity | ≥ 0.45 | Семантическая близость (cosine) |
| style_match | ≥ 0.30 | Соответствие стилю СМИ |
| diversity_score | ≥ 0.70 | Отсутствие дублей среди прогнозов |

In [12]:
import config
from metrics import evaluate_all

metrics_reports = evaluate_all(
    slugs=config.OUTLET_SLUGS,
)



[metrics] ================= STARTING EVALUATION =================

📰 Outlet: KOMMERSANT | Base texts: 346 | Methods: 4



📰 Outlet: LENTA | Base texts: 2887 | Methods: 4



📰 Outlet: INTERFAX | Base texts: 5371 | Methods: 4



[metrics] =================== EVALUATION DONE ===================



In [13]:
# ── Сводная таблица метрик ────────────────────────────────────────
import pandas as pd

TARGETS = {
    "topic_hit_rate":      0.30,
    "entity_match_f1":     0.20,
    "semantic_similarity": 0.45,
    "style_match":         0.30,
    "diversity_score":     0.70,
}

rows = []
for slug, rep in metrics_reports.items():
    row = {"outlet": slug, "method": rep.get("method")}
    for metric, target in TARGETS.items():
        val = rep.get(metric, 0)
        row[metric] = f"{val:.3f}  {'✅' if val >= target else '❌'}"
    rows.append(row)

pd.set_option("display.max_colwidth", 20)
pd.DataFrame(rows).set_index("outlet")

,method,topic_hit_rate,entity_match_f1,semantic_similarity,style_match,diversity_score
outlet,,,,,,
kommersant (inertia),inertia,0.000 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
kommersant (frequency),frequency,0.007 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
kommersant (calendar),calendar,0.000 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
kommersant (llm),llm,0.000 ❌,0.106 ❌,0.419 ❌,0.073 ❌,0.954 ✅
lenta (inertia),inertia,0.035 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
lenta (frequency),frequency,0.037 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
lenta (calendar),calendar,0.000 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅
lenta (llm),llm,0.025 ❌,0.228 ✅,0.399 ❌,0.039 ❌,0.952 ✅
interfax (inertia),inertia,0.000 ❌,0.000 ❌,0.000 ❌,0.000 ❌,1.000 ✅


In [14]:
# ── Визуализация метрик (radar chart) ────────────────────────────
import plotly.graph_objects as go

metric_cols = list(TARGETS.keys())
fig = go.Figure()

for slug, rep in metrics_reports.items():
    values = [rep.get(m, 0) for m in metric_cols]
    values_closed = values + [values[0]]  # close polygon
    cats_closed   = metric_cols + [metric_cols[0]]
    fig.add_trace(go.Scatterpolar(
        r=values_closed,
        theta=cats_closed,
        fill="toself",
        name=f"{config.OUTLETS[rep.get('outlet', slug)]['name']} ({rep.get('method', '?')})",
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Метрики качества прогноза по СМИ",
    showlegend=True,
)
fig.show()

---
## Ячейка 6 — Forecast: прогноз на 02.04.2026

In [15]:
import config, datetime
from forecaster import forecast_all

# ── Настройки ────────────────────────────────────────────────────
TARGET_DATE    = datetime.date(2026, 4, 2)
USE_LLM        = True   # False — пропустить генерацию заголовков через Ollama
FORECAST_SLUGS = config.OUTLET_SLUGS
# ────────────────────────────────────────────────────────────────

forecast_reports = forecast_all(
    slugs=FORECAST_SLUGS,
    target_date=TARGET_DATE,
    use_llm=USE_LLM,
)



[forecast] Коммерсантъ → 2026-04-02
  [llm] kommersant | topic: города / дмитрий / песков / сочи / пресс секретарь...
  [llm] kommersant | topic: сша / ирана / трамп / президент / саудовской...
  [llm] kommersant | topic: области / губернатор / региона / бпла / ударе...

[forecast] Лента.ру → 2026-04-02
  [llm] lenta | topic: украине / трампа / трамп / техника / сша...
  [llm] lenta | topic: сша / ссср / украине / трампа / трамп...
  [llm] lenta | topic: россии / сша / украине / трампа / трамп...

[forecast] Интерфакс → 2026-04-02
  [llm] interfax | topic: трамп / сша / россии / путин / против...
  [llm] interfax | topic: сша / трамп / россии / путин / против...
  [llm] interfax | topic: трамп / сша / ирана / россии / путин...

[forecast] Saved → /Users/justcomex/Documents/nstu/pet/titles_forecating/data/forecasts/forecast_2026-04-02.json


In [16]:
# ── Сводка по каждому СМИ ────────────────────────────────────────
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    name   = rep.get("outlet_name", slug)
    preds  = rep.get("predictions", [])
    llm    = [p for p in preds if p.get("method") == "llm"]
    cal    = [p for p in preds if p.get("method") == "calendar"]
    freq   = [p for p in preds if p.get("method") == "frequency"]
    iner   = [p for p in preds if p.get("method") == "inertia"]

    print(f"\n{'='*60}")
    print(f"  {name} ({slug})")
    print(f"{'='*60}")
    print(f"  Топ-темы: {', '.join(rep.get('top_topics', [])[:3])}")
    print(f"  Контекст событий:\n    {rep.get('events_context', '').replace(chr(10), chr(10)+'    ')}")
    print(f"\n  Baseline — Инерция ({len(iner)} тем):")
    for p in iner[:5]:
        print(f"    · {p.get('topic_label', '')[:70]}")
    print(f"\n  Baseline — Частотность ({len(freq)} тем):")
    for p in freq[:5]:
        print(f"    · {p.get('topic_label', '')[:70]}")
    print(f"\n  Календарь событий ({len(cal)}):")
    for p in cal:
        print(f"    • {p.get('title', '')}")
    if llm:
        print(f"\n  LLM-заголовки ({len(llm)}):")
        for p in llm[:8]:
            print(f"    ▶ {p.get('title', '')}")
            lead = p.get("lead")
            if lead:
                print(f"      {lead[:120]}")


  Коммерсантъ (kommersant)
  Топ-темы: города / дмитрий / песков / сочи / пресс секретарь, сша / ирана / трамп / президент / саудовской, области / губернатор / региона / бпла / ударе
  Контекст событий:
    - 01.04.2026: [holiday] День смеха (1 апреля)
    - 02.04.2026: [economics] Ожидается публикация PMI-индексов промышленного производства (мировые)
    - 02.04.2026: [politics] Заседание Госдумы (плановое)
    - 03.04.2026: [economics] Заседание Банка России / пресс-конференция (ближайшее)

  Baseline — Инерция (0 тем):

  Baseline — Частотность (10 тем):
    · города / дмитрий / песков / сочи / пресс секретарь
    · сша / ирана / трамп / президент / саудовской
    · области / губернатор / региона / бпла / ударе
    · россии / глава / тасс / москве / всу
    · служба / пресс служба / пресс / россии / правительства

  Календарь событий (4):
    • [Шаблон] Коммерсантъ: День смеха (1 апреля)
    • [Шаблон] Коммерсантъ: Ожидается публикация PMI-индексов промышленного производства (миров

---
## Ячейка 7 — Results: просмотр итогового JSON

In [17]:
import json, os
from IPython.display import JSON

json_path = os.path.join(config.FORECASTS_DIR, "forecast_2026-04-02.json")

if os.path.exists(json_path):
    with open(json_path, encoding="utf-8") as f:
        forecast_json = json.load(f)
    print(f"Файл: {json_path}")
    print(f"СМИ в файле: {list(forecast_json.keys())}")
    # Интерактивный JSON-просмотр в Jupyter
    JSON(forecast_json)
else:
    print(f"Файл не найден: {json_path}")
    print("Сначала запустите ячейку Forecast (шаг 6).")

Файл: /Users/justcomex/Documents/nstu/pet/titles_forecating/data/forecasts/forecast_2026-04-02.json
СМИ в файле: ['kommersant', 'lenta', 'interfax']


In [18]:
# ── Таблица всех LLM-прогнозов ────────────────────────────────────
import pandas as pd

rows = []
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    for p in rep.get("predictions", []):
        if p.get("method") == "llm":
            rows.append({
                "СМИ":     config.OUTLETS[slug]["name"],
                "Рубрика": p.get("rubric", ""),
                "Заголовок": p.get("title", ""),
                "Лид":     (p.get("lead") or "")[:120],
            })

if rows:
    df_llm = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 80)
    display(df_llm)
else:
    print("LLM-прогнозы не сгенерированы. Проверьте Ollama или запустите с USE_LLM=True.")

,СМИ,Рубрика,Заголовок,Лид
0,Коммерсантъ,города,Песков прокомментировал подготовку к заседанию Госдумы,"Пресс-секретарь президента Дмитрий Песков заявил, что администрация внимател..."
1,Коммерсантъ,города,Сочи готовится к проведению крупного экономического форума,Администрация курортного города завершила подготовку к международному эконом...
2,Коммерсантъ,города,ЦБ РФ подтвердил дату пресс-конференции по итогам заседания,"Банк России объявил, что пресс-конференция по результатам заседания совета д..."
3,Коммерсантъ,города,PMI промышленности в России показал рост активности,"По предварительным данным, индекс деловой активности в обрабатывающей промыш..."
4,Коммерсантъ,города,Петербургские предприятия увеличили объемы экспортных поставок,Российские компании из Санкт-Петербурга нарастили экспортные операции в перв...
5,Коммерсантъ,сша,Трамп обсудил с саудовским руководством условия нового соглашения по нефти,Администрация США предложила Саудовской Аравии пересмотреть условия экспорта...
6,Коммерсантъ,сша,Иран пригрозил ответом на американские санкции против КСИР,Представитель иранского МИД Аббас Аракчи заявил о готовности Тегерана к асим...
7,Коммерсантъ,сша,Израиль согласовал с США условия авиаударов по иранским объектам,Премьер-министр Израиля Биньямин Нетаньяху получил одобрение администрации Т...
8,Коммерсантъ,сша,Лавров обвинил США в дестабилизации Персидского залива,Министр иностранных дел России Сергей Лавров выступил с критикой американско...
9,Коммерсантъ,сша,Саудовская Аравия усилила контроль над АЭС на фоне региональной напряженности,Королевство развернуло дополнительные системы противовоздушной обороны вокру...


In [20]:
# ── Экспорт прогнозов в Excel ─────────────────────────────────────
import config, datetime, os
import pandas as pd

TARGET_DATE = datetime.date(2026, 4, 2)
EXPORT_PATH = os.path.join(config.FORECASTS_DIR, f"forecast_{TARGET_DATE}.xlsx")

rows = []
for slug, rep in forecast_reports.items():
    if not rep:
        continue
    outlet_name = config.OUTLETS[slug]["name"]
    for p in rep.get("predictions", []):
        rows.append({
            "СМИ":         outlet_name,
            "Метод":       p.get("method", ""),
            "Рубрика":     p.get("rubric", "") or p.get("topic_label", ""),
            "Заголовок":   p.get("title", "") or p.get("topic_label", ""),
            "Лид":         (p.get("lead") or ""),
            "Уверенность": p.get("score", ""),
        })

if rows:
    df_export = pd.DataFrame(rows)

    with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
        # Лист 1: все прогнозы
        df_export.to_excel(writer, sheet_name="Все прогнозы", index=False)

        # Лист 2: только LLM-заголовки
        df_llm = df_export[df_export["Метод"] == "llm"]
        if not df_llm.empty:
            df_llm.to_excel(writer, sheet_name="LLM заголовки", index=False)

        # Лист 3+: по одному листу на СМИ
        for slug, rep in forecast_reports.items():
            if not rep:
                continue
            name = config.OUTLETS[slug]["name"]
            df_outlet = df_export[df_export["СМИ"] == name]
            df_outlet.to_excel(writer, sheet_name=slug[:31], index=False)

    print(f"✅ Файл сохранён: {EXPORT_PATH}")
    print(f"   Строк: {len(df_export)}  |  Листов: {2 + len(forecast_reports)}")
else:
    print("⚠️  Нет данных для экспорта — сначала запустите ячейку Forecast.")


✅ Файл сохранён: /Users/justcomex/Documents/nstu/pet/titles_forecating/data/forecasts/forecast_2026-04-02.xlsx
   Строк: 84  |  Листов: 5


In [21]:
# ── Таблица календарных событий на дату ──────────────────────────
from event_calendar import load_events

events = load_events(target_date=datetime.date(2026, 4, 2), window_days=3)
pd.DataFrame(events)[["date", "event_type", "description", "outlets"]]

,date,event_type,description,outlets
0,2026-04-01,holiday,День смеха (1 апреля),"[rbc, kommersant, lenta, interfax, vedomosti]"
1,2026-04-02,economics,Ожидается публикация PMI-индексов промышленного производства (мировые),"[rbc, vedomosti, kommersant, interfax]"
2,2026-04-02,politics,Заседание Госдумы (плановое),"[rbc, interfax, kommersant]"
3,2026-04-02,economics,Статистика рынка труда США (еженедельная),"[rbc, vedomosti]"
4,2026-04-03,economics,Заседание Банка России / пресс-конференция (ближайшее),"[rbc, vedomosti, kommersant, interfax]"
5,2026-04-04,sport,Тур де Франс / футбольные матчи РПЛ (тур),"[lenta, rbc]"


---
## Быстрый запуск всего пайплайна одной ячейкой

In [ ]:
# !! Выполнит весь пайплайн последовательно !!
# Раскомментируйте нужный вариант
import subprocess, sys

# ВАРИАНТ 1: только прогноз без LLM (быстро, если данные уже собраны)
# subprocess.run([sys.executable, "main.py", "--mode", "forecast",
#                 "--target", "2026-04-02", "--no-llm"], check=True)

# ВАРИАНТ 2: полный пайплайн с LLM (~20-40 мин)
#subprocess.run([sys.executable, "main.py", "--mode", "all",
#                  "--target", "2026-04-02"], check=True)

# ВАРИАНТ 3: только для одного СМИ
# subprocess.run([sys.executable, "main.py", "--mode", "all",
#                 "--outlets", "kommersant", "--target", "2026-04-02"], check=True)

print("Раскомментируйте нужный вариант выше и запустите ячейку.")
